In [7]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
import xgboost as xgb
import pickle

# Data preprocessing function
def preprocess_data(df):
    df = df.copy()
    # Drop columns with too many missing values
    df.drop(columns=["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu"], inplace=True, errors="ignore")
    # Fill missing numerical values
    df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])
    # Fill missing categorical values
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].fillna("None")
    # Feature engineering
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["Remodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)
    df["Qual*Area"] = df["GrLivArea"] * df["OverallQual"]
    # Handle skewness
    numeric_feats = df.select_dtypes(include=np.number).columns
    skewed_feats = df[numeric_feats].apply(lambda x: x.skew()).sort_values(ascending=False)
    high_skew = skewed_feats[skewed_feats > 0.75].index
    df[high_skew] = np.log1p(df[high_skew])
    return df

# Load and preprocess data
train_df = pd.read_csv("/Users/simrannayak/Desktop/Build Project ML Data/Dataset/train (1).csv")
test_df = pd.read_csv("/Users/simrannayak/Desktop/Build Project ML Data/Dataset/test (1).csv")
train_processed = preprocess_data(train_df)
test_processed = preprocess_data(test_df)

# Prepare feature lists
numeric_features = train_processed.select_dtypes(include=np.number).columns.tolist()
categorical_features = train_processed.select_dtypes(include="object").columns.tolist()
if "SalePrice" in numeric_features:
    numeric_features.remove("SalePrice")

# Preprocessor pipeline (fixed sparse=True compatible with sklearn 0.24.2)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('transformer', PowerTransformer(method='yeo-johnson'))
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # sparse_output not supported
        ]), categorical_features)
    ])

# Prepare train/test splits
X = train_processed.drop(columns=["SalePrice"])
y = np.log1p(train_df["SalePrice"])
X_test = test_processed

print("Fitting preprocessor...")
preprocessor.fit(X)
X_transformed = preprocessor.transform(X)
X_test_transformed = preprocessor.transform(X_test)

# ✅ Get feature names with get_feature_names() for sklearn 0.24.2
cat_feature_names = preprocessor.named_transformers_['cat']\
                                .named_steps['encoder']\
                                .get_feature_names_out(categorical_features)
preprocessed_feature_names = numeric_features + cat_feature_names.tolist()

# Random Forest for feature importance
print("Fitting Random Forest for feature selection...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_transformed, y)

# Select top 30 features
feature_selector = SelectFromModel(rf_model, max_features=30, threshold=-np.inf, prefit=True)
X_selected = feature_selector.transform(X_transformed)
X_test_selected = feature_selector.transform(X_test_transformed)

# Extract selected feature names
selected_mask = feature_selector.get_support()
selected_features = [preprocessed_feature_names[i] for i in range(len(preprocessed_feature_names)) if selected_mask[i]]

print("Top 30 features selected:")
feature_importance_df = pd.DataFrame({
    'feature': preprocessed_feature_names,
    'importance': rf_model.feature_importances_
})
print(feature_importance_df.nlargest(30, 'importance'))

# Create XGBoost pipeline (data already preprocessed)
xgb_model = Pipeline(steps=[
    ('regressor', xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
xgb_scores = -cross_val_score(xgb_model, X_selected, y, cv=kf, scoring="neg_mean_squared_error")
print(f"XGBoost CV Score with top 30 features: {np.mean(xgb_scores):.5f}")

# Train on full dataset
xgb_model.fit(X_selected, y)

# Predict on test set
xgb_pred = np.expm1(xgb_model.predict(X_test_selected))

# Export predictions
output = pd.DataFrame({"Id": test_df["Id"], "SalePrice": xgb_pred})
output.to_csv("xgboost_predictions_rf_selection.csv", index=False)

# Save the trained model, preprocessor, and selected features
def save_model_and_features(model, preprocessor, selected_features):
    with open('xgboost_model_rf_selection.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open('preprocessor.pkl', 'wb') as f:
        pickle.dump(preprocessor, f)
    with open('selected_features.pkl', 'wb') as f:
        pickle.dump(selected_features, f)

# Save the model and features
save_model_and_features(xgb_model, preprocessor, selected_features)


Fitting preprocessor...
Fitting Random Forest for feature selection...
Top 30 features selected:
              feature  importance
41          Qual*Area    0.581985
37            TotalSF    0.119191
4         OverallQual    0.049144
12        TotalBsmtSF    0.018884
26         GarageCars    0.017102
9          BsmtFinSF1    0.016093
39                Age    0.012886
7        YearRemodAdd    0.010281
3             LotArea    0.009779
6           YearBuilt    0.008832
27         GarageArea    0.008747
38          TotalBath    0.008389
5         OverallCond    0.008103
46        MSZoning_RM    0.007487
226      CentralAir_N    0.006737
13           1stFlrSF    0.006073
11          BsmtUnfSF    0.005035
16          GrLivArea    0.004794
25        GarageYrBlt    0.004620
2         LotFrontage    0.004424
227      CentralAir_Y    0.004065
14           2ndFlrSF    0.003717
0                  Id    0.003641
237    KitchenQual_TA    0.003029
29        OpenPorchSF    0.002734
35             MoSo